# MiniGPT — a GPT-style language model built from scratch (Tiny Shakespeare)

This notebook reproduces the project end to end from a clean Colab runtime:

1. clone the repository and install dependencies
2. download and split the dataset
3. run the unit tests (attention, causal mask, shapes, tiny-overfit, checkpoint reload, API)
4. reproduce the architecture checks (hand-calculated attention, mask, multi-head shape trace, positional encoding)
5. **A:** download the released checkpoint and generate text — **B:** (optional) train the model yourself
6. evaluate, plot the loss curves and inspect attention
7. call the REST API

`Runtime -> Change runtime type -> T4 GPU` is recommended (everything except training also works on CPU).

In [ ]:
!nvidia-smi -L || echo "no GPU - CPU runtime"

## 1. Clone and install

In [ ]:
!git clone -q https://github.com/AbdulRahmanAzam/minigpt-shakespeare.git
%cd minigpt-shakespeare
!pip install -q -r requirements.txt
import torch; print("torch", torch.__version__, "| cuda", torch.cuda.is_available())

## 2. Dataset (Tiny Shakespeare, 90/5/5 split identical to the Hugging Face loading script)

In [ ]:
!python scripts/prepare_data.py

## 3. Unit tests

In [ ]:
!python -m pytest -q

## 4. Architecture checks (Figures 1–5)

In [ ]:
!python -m experiments.hand_attention

In [ ]:
!python -m experiments.mask_visualization
!python -m experiments.shape_trace
!python -m experiments.positional_encoding_viz
!python -m experiments.model_summary --config configs/tiny_gpt.yaml

In [ ]:
from IPython.display import Image, display
for f in ["fig1_hand_attention", "fig2_causal_mask", "fig4_positional_encoding"]:
    display(Image(f"outputs/figures/{f}.png", width=900))

## 5A. Use the released checkpoint

The trained weights are published as a GitHub Release asset (kept out of git on purpose).

In [ ]:
!python scripts/download_checkpoint.py
!python -m src.generate --prompt "ROMEO:" --max-new-tokens 300 --temperature 0.8 --seed 42

In [ ]:
for prompt in ["JULIET:", "First Citizen:", "To be, or not to be"]:
    !python -m src.generate --prompt "{prompt}" --max-new-tokens 200 --seed 7

## 5B. (Optional) Train it yourself

The tiny overfit gate first (should drop from ~4.2 to near 0), then the full model.
The full run is 5000 steps (≈ 15 min on an RTX 3050 laptop GPU; similar on a Colab T4, which uses fp16 automatically).
Set `FULL_TRAINING = True` to run it; the new checkpoint lands in `checkpoints/tiny_gpt/best.pt`.

In [ ]:
!python -m src.train --config configs/overfit.yaml

In [ ]:
FULL_TRAINING = False
if FULL_TRAINING:
    !python -m src.train --config configs/tiny_gpt.yaml
    !python -m experiments.plot_training --run outputs/logs/tiny_gpt

## 6. Evaluation, loss curves and attention

In [ ]:
CKPT = "checkpoints/tiny_gpt/best.pt" if FULL_TRAINING else "checkpoints/minigpt_shakespeare.pt"
!python -m experiments.baselines
!python -m experiments.evaluate --checkpoint {CKPT}
!python -m experiments.attention_visualization --checkpoint {CKPT}

In [ ]:
for f in ["fig6_loss_curves", "fig7_attention_detail", "fig9_ablation"]:
    try:
        display(Image(f"outputs/figures/{f}.png", width=900))
    except FileNotFoundError:
        print("missing", f)

## 7. REST API (FastAPI)

In [ ]:
import subprocess, time, requests
server = subprocess.Popen(["uvicorn", "src.api:app", "--port", "8000"])
for _ in range(60):
    try:
        if requests.get("http://127.0.0.1:8000/health").ok: break
    except requests.ConnectionError:
        time.sleep(1)
print(requests.get("http://127.0.0.1:8000/model-info").json()["parameters"], "parameters")
r = requests.post("http://127.0.0.1:8000/generate",
                  json={"prompt": "ROMEO:", "max_new_tokens": 200, "temperature": 0.8, "seed": 1})
print(r.json()["text"])
server.terminate()